# DiD

In [ ]:
%run ./did_model

In [ ]:
# Settings
match_method = "summary_4"

OUTCOME_COLS = {
    "H": "high_price_peak",       
    "L": "low_price_peak",        
    "T": "top3_mean_consumption"  
}

EVENT_WINDOW = (-12, 12)
REFERENCE_EVENT_TIME = -1

In [ ]:
# Read data
matches = pd.read_parquet(f"/lakehouse/default/Files/output/matching/{match_method}/matches")
month_result = pd.read_parquet("/lakehouse/default/Files/month_data")

# Build matched panel
df = build_matched_panel(
    matches=matches,
    month_result=month_result,
    outcome_cols=OUTCOME_COLS
)

df.head()

In [ ]:
# Run baseline DiD
basic_models, basic_results_df = run_basic_did_all(
    df=df,
    outcome_cols=OUTCOME_COLS
)

basic_results_df

In [ ]:
cohort_basic_models, cohort_basic_results_df = run_basic_did_by_cohort(
    df=df,
    outcome_cols=OUTCOME_COLS,
    min_treated=30,
    cluster_col="aID"
)

cohort_basic_results_df

In [ ]:
# Run event-study for H, L, T
event_models, event_results_df = run_event_study_all(
    df=df,
    outcome_cols=OUTCOME_COLS,
    event_window=EVENT_WINDOW,
    reference=REFERENCE_EVENT_TIME
)

event_results_df

In [ ]:
cohort_event_models, cohort_event_results_df = run_event_study_by_cohort(
    df=df,
    outcome_cols=OUTCOME_COLS,
    event_window=(-12, 6),
    reference=-1,
    min_treated=30
)
cohort_event_results_df


In [ ]:
plot_event_study_by_cohort(
    cohort_event_results_df,
    outcome_name="H",
    title="Cohort-specific event-study: High-price peak"
)

In [ ]:
plot_event_study_by_cohort(
    cohort_event_results_df,
    outcome_name="L",
    title="Cohort-specific event-study: Low-price peak"
)

In [ ]:
plot_event_study_by_cohort(
    cohort_event_results_df,
    outcome_name="T",
    title="Cohort-specific event-study: Overall peak"
)

In [ ]:
# Plot event-study results
plot_event_study(event_results_df, "H", "Event-study: High-price-period peak consumption")
plot_event_study(event_results_df, "L", "Event-study: Low-price-period peak consumption")
plot_event_study(event_results_df, "T", "Event-study: Overall peak consumption")

In [ ]:
# Save results
output_dir = f"/lakehouse/default/Files/output/did/{match_method}"

basic_results_df.to_parquet(f"{output_dir}/basic_did_results.parquet", index=False)
event_results_df.to_parquet(f"{output_dir}/event_study_results.parquet", index=False)